# Compare RWKV-MedSAM2, Medical SAM2, MedSAM2, and SAM2.1

Benchmark all available models on the same test split with Oxford-paper-style metrics. Primary metrics: Dice, IoU, and HD95. Supplementary metrics: false-positive/false-negative behavior, foreground/empty-mask behavior, and prediction efficiency.

Prompt protocols: `box`, `mask`, and `mixed`. Click prompts are intentionally excluded.

Paper references:
- Oxford Medical SAM 2: https://arxiv.org/abs/2408.00874
- Oxford HTML text: https://ar5iv.labs.arxiv.org/html/2408.00874


## 0. User Settings

Edit paths here first. Missing checkpoints are allowed: preflight will mark them as `pending` instead of failing the notebook.

In [ ]:
import os
import sys
import gc
import csv
import json
import time
import math
import glob
import pickle
import random
import inspect
import traceback
from pathlib import Path
from dataclasses import dataclass, asdict
from collections import defaultdict, Counter

import numpy as np
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
from IPython.display import display

try:
    import pandas as pd
except Exception:
    pd = None

try:
    import SimpleITK as sitk
except Exception:
    sitk = None

try:
    from scipy.ndimage import distance_transform_edt, binary_erosion
except Exception:
    distance_transform_edt = None
    binary_erosion = None

SEED = 42
TARGET_HW = 512
LOADERS_PKL = "/data/loaders32.pkl"
EXCLUDED_DATASET = "wanglab__CT_DeepLesion-MedSAM2"

# Full benchmark knobs. Start small for smoke/debug, then set MAX_TEST_CASES = None.
MAX_TEST_CASES = None
SMOKE_CASES_PER_DIM = 1
PROMPT_MODES = ["box", "mask", "mixed"]
MIXED_PROMPT_PROBS = {"box": 0.5, "mask": 0.5}
THRESHOLDS = np.linspace(0.05, 0.95, 19, dtype=np.float32)
PRIMARY_THRESHOLD = 0.5

RUN_ROOT = Path("/data/rwkv_medsam2_model_comparisons")
RUN_NAME = time.strftime("compare_oxford_metrics_%Y%m%d_%H%M%S")
OUT_DIR = RUN_ROOT / RUN_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

REPO_ROOT_CANDIDATES = [
    os.getcwd(),
    "/RWKV-MedSAM2",
    "/data/jrbonill/RWKV-MedSAM2",
    "/data/jrbonill/rwkv_medsam2",
    "E:/dev/RWKV-MedSAM2",
]

OXFORD_ENV_BASE = "/data/jrbonill/oxford_medsam2_env"
OXFORD_REPO = f"{OXFORD_ENV_BASE}/Medical-SAM2"
OXFORD_SITE = f"{OXFORD_ENV_BASE}/site"
OXFORD_SAM_CKPT = f"{OXFORD_REPO}/checkpoints/sam2_hiera_tiny.pt"
OXFORD_MED_PRETRAIN = f"{OXFORD_REPO}/checkpoints/MedSAM2_pretrain.pth"

UOFT_MEDSAM2_BASE = "/data/jrbonill/medsam2_env/MedSAM2"
UOFT_MEDSAM2_SITE = "/data/jrbonill/medsam2_env/site"
UOFT_MEDSAM2_CKPT = f"{UOFT_MEDSAM2_BASE}/checkpoints/MedSAM2_latest.pt"
UOFT_MEDSAM2_CFG = "sam2.1_hiera_t512.yaml"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)
print("OUT_DIR:", OUT_DIR)


## 1. Environment, Data, and Shared Utilities

In [ ]:
def set_determinism(seed=SEED):
    random.seed(int(seed))
    np.random.seed(int(seed))
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception:
        pass

set_determinism(SEED)


def find_repo_root():
    for root in REPO_ROOT_CANDIDATES:
        root = os.path.abspath(str(root))
        if os.path.isfile(os.path.join(root, "rwkv_medsam2", "train_sam2.py")):
            return root
    raise FileNotFoundError(f"Could not find RWKV-MedSAM2 repo from: {REPO_ROOT_CANDIDATES}")

REPO_ROOT = find_repo_root()
EXT_ROOT = os.path.join(REPO_ROOT, "ext")
os.chdir(REPO_ROOT)
for p in [REPO_ROOT, EXT_ROOT]:
    if p not in sys.path:
        sys.path.insert(0, p)
print("REPO_ROOT:", REPO_ROOT)

from hydra.core.global_hydra import GlobalHydra
from hydra import initialize_config_dir
from omegaconf import OmegaConf

from rwkv_medsam2.train_sam2 import load_config, get_data_loaders, build_student_predictor, collate_sam2_batch
from rwkv_medsam2.functions.func_metrics import dice_iou


def load_saved_loaders_robust(pkl_path):
    with open(pkl_path, "rb") as f:
        obj = pickle.load(f)
    if isinstance(obj, (tuple, list)) and len(obj) == 3:
        return obj[0], obj[1], obj[2]
    if isinstance(obj, dict):
        key_sets = [
            ("train_loader", "val_loader", "test_loader"),
            ("train_loaders", "val_loaders", "test_loaders"),
            ("train", "val", "test"),
        ]
        for keys in key_sets:
            if all(k in obj for k in keys):
                return obj[keys[0]], obj[keys[1]], obj[keys[2]]
        loader_like = [v for v in obj.values() if hasattr(v, "dataset") and hasattr(v, "__iter__")]
        if len(loader_like) >= 3:
            return loader_like[:3]
    raise RuntimeError(f"Unsupported loader pickle format: {pkl_path}")


def apply_dataset_exclusion(loader, excluded_dataset=EXCLUDED_DATASET):
    if loader is None or not hasattr(loader, "dataset") or not hasattr(loader.dataset, "sequences"):
        return {"before": None, "after": None, "removed": 0}
    before = len(loader.dataset.sequences)
    kept = [s for s in loader.dataset.sequences if str(s.get("dataset", "")).strip() != excluded_dataset]
    removed = before - len(kept)
    loader.dataset.sequences = kept
    if hasattr(loader.dataset, "entry_dims"):
        loader.dataset.entry_dims = [s.get("dim", 2) for s in kept]
    return {"before": before, "after": len(kept), "removed": removed}


def load_benchmark_loaders():
    if os.path.exists(LOADERS_PKL):
        print("Loading saved loaders:", LOADERS_PKL)
        train_loader, val_loader, test_loader = load_saved_loaders_robust(LOADERS_PKL)
    else:
        print("Saved loaders missing; rebuilding from VCR config.")
        cfg = load_config(os.path.join(REPO_ROOT, "ext", "sam2", "configs", "sam2.1", "sam2.1_vcr.yaml"))
        train_loader, val_loader, test_loader = get_data_loaders(cfg)
    for name, loader in [("train", train_loader), ("val", val_loader), ("test", test_loader)]:
        print(name, apply_dataset_exclusion(loader))
    return train_loader, val_loader, test_loader

train_loader, val_loader, test_loader = load_benchmark_loaders()
benchmark_loader = test_loader
print("Test batches:", len(benchmark_loader))


def first_cases_by_dim(loader, n_per_dim=1):
    out = {2: [], 3: []}
    ds = loader.dataset
    for i in range(len(ds)):
        try:
            item = ds[i]
            dim = int(item.get("dim", 2))
            if dim in out and len(out[dim]) < n_per_dim:
                out[dim].append(i)
            if all(len(v) >= n_per_dim for v in out.values()):
                break
        except Exception:
            continue
    return out

print("Smoke dataset indices by dim:", first_cases_by_dim(benchmark_loader, SMOKE_CASES_PER_DIM))


## 2. Metrics

Primary Oxford metrics are Dice, IoU, and HD95. Dice and IoU are reported at `PRIMARY_THRESHOLD`; threshold-sweep columns are aggregated across cases to select the best group-level thresholds, rather than choosing a threshold per image. The extra metrics make error modes and deployment cost explicit.


In [ ]:
def sigmoid_np(x):
    x = np.asarray(x, dtype=np.float32)
    x = np.clip(x, -30.0, 30.0)
    return 1.0 / (1.0 + np.exp(-x))


def ensure_thw(x):
    if torch.is_tensor(x):
        x = x.detach().cpu().numpy()
    x = np.asarray(x)
    while x.ndim > 3 and 1 in x.shape:
        x = np.squeeze(x)
    if x.ndim == 2:
        x = x[None, ...]
    if x.ndim != 3:
        raise ValueError(f"Expected [T,H,W] or [H,W], got shape {x.shape}")
    return x


def confusion_counts(pred, gt):
    p = np.asarray(pred).astype(bool)
    g = np.asarray(gt).astype(bool)
    tp = int(np.logical_and(p, g).sum())
    fp = int(np.logical_and(p, ~g).sum())
    fn = int(np.logical_and(~p, g).sum())
    tn = int(np.logical_and(~p, ~g).sum())
    return tp, fp, fn, tn


def safe_div(num, den, default=float("nan")):
    den = float(den)
    return float(num) / den if den != 0 else default


def threshold_suffix(thr):
    return f"{float(thr):.2f}".replace(".", "p")


def threshold_metric_key(metric, thr):
    return f"{metric}_thr_{threshold_suffix(thr)}"


def metric_at_threshold(row, metric, thr):
    return row.get(threshold_metric_key(metric, thr), float("nan"))


def hd95_binary(pred, gt, spacing=None):
    pred = np.asarray(pred).astype(bool)
    gt = np.asarray(gt).astype(bool)
    if pred.shape != gt.shape:
        raise ValueError(f"HD95 shape mismatch: {pred.shape} vs {gt.shape}")
    if not pred.any() and not gt.any():
        return 0.0
    if pred.any() != gt.any():
        return float("nan")
    if distance_transform_edt is None or binary_erosion is None:
        return float("nan")
    if spacing is None:
        spacing = (1.0,) * pred.ndim
    pred_border = np.logical_xor(pred, binary_erosion(pred))
    gt_border = np.logical_xor(gt, binary_erosion(gt))
    if not pred_border.any() or not gt_border.any():
        return 0.0 if np.array_equal(pred, gt) else float("nan")
    dt_pred = distance_transform_edt(~pred_border, sampling=spacing)
    dt_gt = distance_transform_edt(~gt_border, sampling=spacing)
    distances = np.concatenate([dt_gt[pred_border], dt_pred[gt_border]]).astype(np.float64)
    if distances.size == 0:
        return float("nan")
    return float(np.percentile(distances, 95))


def segmentation_metrics_from_logits(logits_thw, gt_thw, thresholds=THRESHOLDS):
    logits = ensure_thw(logits_thw).astype(np.float32)
    gt = (ensure_thw(gt_thw) > 0).astype(np.uint8)
    if logits.shape != gt.shape:
        raise ValueError(f"Metric shape mismatch: logits {logits.shape} vs gt {gt.shape}")
    probs = sigmoid_np(logits)

    pred05 = (probs >= PRIMARY_THRESHOLD).astype(np.uint8)
    tp, fp, fn, tn = confusion_counts(pred05, gt)
    dice05 = safe_div(2 * tp, 2 * tp + fp + fn, default=1.0 if gt.sum() == 0 and pred05.sum() == 0 else 0.0)
    iou05 = safe_div(tp, tp + fp + fn, default=1.0 if gt.sum() == 0 and pred05.sum() == 0 else 0.0)

    threshold_metrics = {}
    dice_vals, iou_vals = [], []
    for thr in thresholds:
        thr = float(thr)
        pred = (probs >= thr).astype(np.uint8)
        ttp, ffp, ffn, ttn = confusion_counts(pred, gt)
        d = safe_div(2 * ttp, 2 * ttp + ffp + ffn, default=1.0 if gt.sum() == 0 and pred.sum() == 0 else 0.0)
        i = safe_div(ttp, ttp + ffp + ffn, default=1.0 if gt.sum() == 0 and pred.sum() == 0 else 0.0)
        threshold_metrics[threshold_metric_key("dice", thr)] = float(d)
        threshold_metrics[threshold_metric_key("iou", thr)] = float(i)
        dice_vals.append(d)
        iou_vals.append(i)

    hd95_2d = []
    for t in range(gt.shape[0]):
        hd95_2d.append(hd95_binary(pred05[t], gt[t]))
    finite_hd = [x for x in hd95_2d if not (math.isnan(x) or math.isinf(x))]

    metrics = {
        "dice": float(dice05),
        "iou": float(iou05),
        "hd95": float(np.mean(finite_hd)) if finite_hd else float("nan"),
        "dice_avg_threshold": float(np.mean(dice_vals)),
        "iou_avg_threshold": float(np.mean(iou_vals)),
        "tp": tp, "fp": fp, "fn": fn, "tn": tn,
        "false_negative_rate": safe_div(fn, tp + fn, default=0.0),
        "false_positive_rate": safe_div(fp, fp + tn, default=0.0),
        "precision_ppv": safe_div(tp, tp + fp, default=1.0 if pred05.sum() == 0 and gt.sum() == 0 else 0.0),
        "recall_sensitivity": safe_div(tp, tp + fn, default=1.0 if gt.sum() == 0 else 0.0),
        "specificity": safe_div(tn, tn + fp, default=0.0),
        "volumetric_similarity": 1.0 - safe_div(abs(int(pred05.sum()) - int(gt.sum())), int(pred05.sum()) + int(gt.sum()), default=0.0),
        "pred_fg_frac": float(pred05.mean()),
        "gt_fg_frac": float(gt.mean()),
        "empty_pred": float(pred05.sum() == 0),
        "empty_gt": float(gt.sum() == 0),
        "missed_object": float((gt.sum() > 0) and (pred05.sum() == 0)),
        "hallucination": float((gt.sum() == 0) and (pred05.sum() > 0)),
        "n_frames": int(gt.shape[0]),
    }
    metrics.update(threshold_metrics)
    if gt.ndim == 3:
        metrics["hd95_3d"] = hd95_binary(pred05, gt)
    return metrics


def threshold_score_is_better(score, best_score, thr, best_thr):
    if math.isnan(score):
        return False
    if math.isnan(best_score) or score > best_score:
        return True
    if np.isclose(score, best_score):
        return abs(float(thr) - PRIMARY_THRESHOLD) < abs(float(best_thr) - PRIMARY_THRESHOLD)
    return False


def add_best_threshold_columns(row, thresholds=THRESHOLDS):
    best_dice_score, best_dice_thr = float("nan"), float(PRIMARY_THRESHOLD)
    best_iou_score, best_iou_thr = float("nan"), float(PRIMARY_THRESHOLD)
    for thr in thresholds:
        thr = float(thr)
        dice_score = float(row.get(threshold_metric_key("dice", thr), float("nan")))
        iou_score = float(row.get(threshold_metric_key("iou", thr), float("nan")))
        if threshold_score_is_better(dice_score, best_dice_score, thr, best_dice_thr):
            best_dice_score, best_dice_thr = dice_score, thr
        if threshold_score_is_better(iou_score, best_iou_score, thr, best_iou_thr):
            best_iou_score, best_iou_thr = iou_score, thr

    row["best_dice_threshold"] = float(best_dice_thr)
    row["dice_at_best_dice_threshold"] = float(best_dice_score)
    row["iou_at_best_dice_threshold"] = float(row.get(threshold_metric_key("iou", best_dice_thr), float("nan")))
    row["best_iou_threshold"] = float(best_iou_thr)
    row["iou_at_best_iou_threshold"] = float(best_iou_score)
    row["dice_at_best_iou_threshold"] = float(row.get(threshold_metric_key("dice", best_iou_thr), float("nan")))
    return row


def aggregate_records(records, group_keys):
    buckets = defaultdict(list)
    for r in records:
        key = tuple(r.get(k, "") for k in group_keys)
        buckets[key].append(r)
    rows = []
    numeric_skip = {"tp", "fp", "fn", "tn"}
    for key, vals in buckets.items():
        row = {k: v for k, v in zip(group_keys, key)}
        row["n_cases"] = len(vals)
        metric_keys = sorted({mk for v in vals for mk in v.keys() if isinstance(v.get(mk), (int, float, np.integer, np.floating))})
        for mk in metric_keys:
            xs = [float(v[mk]) for v in vals if mk in v and isinstance(v[mk], (int, float, np.integer, np.floating)) and not math.isnan(float(v[mk]))]
            if not xs:
                continue
            if mk in numeric_skip:
                row[mk] = float(np.sum(xs))
            else:
                row[mk] = float(np.mean(xs))
        add_best_threshold_columns(row)
        rows.append(row)
    return rows


def threshold_sweep_records(records, group_keys, thresholds=THRESHOLDS):
    buckets = defaultdict(list)
    for r in records:
        key = tuple(r.get(k, "") for k in group_keys)
        buckets[key].append(r)

    rows = []
    for key, vals in buckets.items():
        base = {k: v for k, v in zip(group_keys, key)}
        for thr in thresholds:
            thr = float(thr)
            dice_key = threshold_metric_key("dice", thr)
            iou_key = threshold_metric_key("iou", thr)
            dice_vals = [float(v[dice_key]) for v in vals if dice_key in v and not math.isnan(float(v[dice_key]))]
            iou_vals = [float(v[iou_key]) for v in vals if iou_key in v and not math.isnan(float(v[iou_key]))]
            rows.append({
                **base,
                "threshold": thr,
                "n_cases": len(vals),
                "dice": float(np.mean(dice_vals)) if dice_vals else float("nan"),
                "iou": float(np.mean(iou_vals)) if iou_vals else float("nan"),
            })
    return rows


def write_table(rows, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if pd is not None:
        pd.DataFrame(rows).to_csv(path, index=False)
    else:
        keys = sorted({k for r in rows for k in r.keys()})
        with open(path, "w", newline="") as f:
            if keys:
                w = csv.DictWriter(f, fieldnames=keys)
                w.writeheader()
                w.writerows(rows)
            else:
                f.write("")
    print("wrote", path)


## 3. Prompt Generation

Prompts are generated once per sample and reused for every model.

In [ ]:
def bbox_from_mask_np(mask_hw, pad=5):
    mask = np.asarray(mask_hw) > 0
    ys, xs = np.where(mask)
    H, W = mask.shape[-2:]
    if len(xs) == 0:
        return np.array([0, 0, W - 1, H - 1], dtype=np.float32)
    x0 = max(0, int(xs.min()) - pad)
    y0 = max(0, int(ys.min()) - pad)
    x1 = min(W - 1, int(xs.max()) + pad)
    y1 = min(H - 1, int(ys.max()) + pad)
    return np.array([x0, y0, x1, y1], dtype=np.float32)


def sample_to_case(sample, dataset_index=None):
    image = sample["image"]
    mask = sample["mask"]
    if torch.is_tensor(image):
        image_t = image.detach().float()
    else:
        image_t = torch.as_tensor(image).float()
    if torch.is_tensor(mask):
        mask_t = mask.detach().float()
    else:
        mask_t = torch.as_tensor(mask).float()
    if image_t.ndim == 3:
        image_t = image_t.unsqueeze(0)
    if mask_t.ndim == 3:
        mask_t = mask_t.unsqueeze(1)
    T = int(image_t.shape[0])
    valid = sample.get("valid", torch.ones(T, dtype=torch.bool))
    if torch.is_tensor(valid):
        valid_t = valid.detach().bool().flatten()
    else:
        valid_t = torch.as_tensor(valid).bool().flatten()
    valid_t = valid_t[:T]
    gt_thw = mask_t[:, 0].detach().cpu().numpy() > 0.5
    boxes = np.stack([bbox_from_mask_np(gt_thw[t]) for t in range(T)], axis=0)
    meta = {
        "dataset_index": dataset_index,
        "dataset": sample.get("dataset", "unknown"),
        "subdataset": sample.get("subdataset", "default"),
        "task_id": sample.get("task_id", sample.get("task_label", "unknown")),
        "task_label": sample.get("task_label", sample.get("task_id", "unknown")),
        "modality": sample.get("modality", "unknown"),
        "dim": int(sample.get("dim", 2)),
        "seq_idx": sample.get("seq_idx", dataset_index),
    }
    return {"image": image_t, "mask": mask_t, "valid": valid_t, "gt_thw": gt_thw, "boxes": boxes, "meta": meta}


def prompt_plan_for_case(case, prompt_mode, seed=SEED):
    T = int(case["image"].shape[0])
    gt = case["gt_thw"]
    valid = case["valid"].detach().cpu().numpy().astype(bool)
    nonempty = gt.reshape(T, -1).sum(axis=1) > 0
    eligible = np.where(valid & nonempty)[0]
    if len(eligible) == 0:
        eligible = np.where(valid)[0]
    if len(eligible) == 0:
        eligible = np.arange(T)
    # For 3D, prompt one representative foreground frame; for 2D T is normally one.
    areas = gt.reshape(T, -1).sum(axis=1)
    prompt_frame = int(eligible[np.argmax(areas[eligible])]) if len(eligible) else 0
    rng = np.random.RandomState((int(seed) * 1000003 + int(case["meta"].get("dataset_index") or 0) * 97) % (2**32 - 1))
    if prompt_mode == "mixed":
        selected = rng.choice(["box", "mask"], p=[MIXED_PROMPT_PROBS["box"], MIXED_PROMPT_PROBS["mask"]]).item()
    elif prompt_mode in ("box", "mask"):
        selected = prompt_mode
    else:
        raise ValueError(f"Unsupported prompt mode {prompt_mode}; clicks are intentionally disabled.")
    return {
        "requested_prompt_mode": prompt_mode,
        "effective_prompt_mode": selected,
        "prompt_frames": [prompt_frame],
        "boxes": case["boxes"].astype(np.float32),
        "mask_prompts": case["gt_thw"].astype(np.float32),
    }


## 4. Model Registry and Adapters

In [ ]:
@dataclass
class ModelEntry:
    name: str
    family: str
    config_path: str = ""
    checkpoint_dir: str = ""
    checkpoint_path: str = ""
    checkpoint_policy: str = "best_or_latest"
    builder: str = ""
    enabled: bool = True
    status: str = "unknown"
    notes: str = ""


def default_model_registry():
    return [
        ModelEntry(
            name="rwkv_medsam2_distill",
            family="rwkv",
            config_path=os.path.join(REPO_ROOT, "ext/sam2/configs/sam2.1/sam2.1_vcr.yaml"),
            checkpoint_dir=os.path.join(REPO_ROOT, "checkpoints/sam2.1_vcr.pth"),
            builder="rwkv_student",
            notes="RWKV-MedSAM2 with distillation."
        ),
        ModelEntry(
            name="rwkv_medsam2_nodistill",
            family="rwkv",
            config_path=os.path.join(REPO_ROOT, "ext/sam2/configs/sam2.1/sam2.1_vcr_nodistill.yaml"),
            checkpoint_dir=os.path.join(REPO_ROOT, "checkpoints/sam2.1_vcr_nodistill.pth"),
            builder="rwkv_student",
            notes="Pending remote training checkpoint."
        ),
        ModelEntry(
            name="sam2_1_base",
            family="rwkv_base_config",
            config_path=os.path.join(REPO_ROOT, "ext/sam2/configs/sam2.1/sam2.1_hiera_t512.yaml"),
            checkpoint_dir=os.path.join(REPO_ROOT, "checkpoints/sam2.1_hiera_t512_base.pth"),
            builder="sam21_base_student",
            notes="Pending remote training checkpoint; config path follows training_sam2.1_base.ipynb."
        ),
        ModelEntry(
            name="oxford_medical_sam2",
            family="oxford",
            config_path="sam2_hiera_t",
            checkpoint_path=OXFORD_MED_PRETRAIN,
            checkpoint_dir=os.path.dirname(OXFORD_MED_PRETRAIN),
            builder="oxford_video_predictor",
            notes="Uses Oxford-Refine_v9.ipynb loading pattern."
        ),
        ModelEntry(
            name="uoft_medsam2",
            family="uoft",
            config_path=UOFT_MEDSAM2_CFG,
            checkpoint_path=UOFT_MEDSAM2_CKPT,
            checkpoint_dir=os.path.dirname(UOFT_MEDSAM2_CKPT),
            builder="uoft_medsam2_build_sam2",
            notes="Uses MedSAM2-Setup.ipynb and MedSAM2-Refine_v2.ipynb loading pattern."
        ),
    ]

MODEL_REGISTRY = default_model_registry()
for entry in MODEL_REGISTRY:
    print(asdict(entry))


def pick_checkpoint(entry: ModelEntry):
    if entry.checkpoint_path and os.path.isfile(entry.checkpoint_path):
        return entry.checkpoint_path
    if not entry.checkpoint_dir or not os.path.isdir(entry.checkpoint_dir):
        return None
    candidates = []
    for name in ["best.pth", "best_val.pth", "latest.pth"]:
        p = os.path.join(entry.checkpoint_dir, name)
        if os.path.isfile(p):
            candidates.append(p)
    candidates.extend(sorted(glob.glob(os.path.join(entry.checkpoint_dir, "epoch_*.pth"))))
    if not candidates:
        return None
    if entry.checkpoint_policy == "latest":
        return max(candidates, key=os.path.getmtime)
    for p in candidates:
        if os.path.basename(p) in ("best.pth", "best_val.pth"):
            return p
    return max(candidates, key=os.path.getmtime)


def unwrap_state_dict(obj):
    if isinstance(obj, dict):
        for key in ["model", "state_dict", "model_state_dict", "net", "network"]:
            if key in obj and isinstance(obj[key], dict):
                return obj[key]
    return obj


def load_rwkv_config(entry: ModelEntry):
    if entry.builder == "sam21_base_student":
        cfg = load_config(os.path.join(REPO_ROOT, "ext/sam2/configs/sam2.1/sam2.1_vcr.yaml"))
        base_cfg = OmegaConf.load(entry.config_path)
        cfg.model = base_cfg.model
        cfg._config_path = "sam2.1/sam2.1_hiera_t512.yaml"
        cfg.ckpt_path = entry.checkpoint_dir
        cfg.logging.filename = "sam2.1_hiera_t512_base.log"
        cfg.init.ignore_prefixes = []
        return cfg
    return load_config(entry.config_path)


def load_rwkv_model(entry: ModelEntry):
    ckpt_path = pick_checkpoint(entry)
    if ckpt_path is None:
        raise FileNotFoundError(f"No checkpoint yet for {entry.name}: {entry.checkpoint_dir}")
    cfg = load_rwkv_config(entry)
    cfg.training.device = str(DEVICE)
    model = build_student_predictor(cfg).to(DEVICE)
    ckpt = torch.load(ckpt_path, map_location="cpu")
    sd = unwrap_state_dict(ckpt)
    missing, unexpected = model.load_state_dict(sd, strict=False)
    model.eval()
    return {"entry": entry, "model": model, "config": cfg, "checkpoint": ckpt_path, "missing": len(missing), "unexpected": len(unexpected)}


def load_oxford_model(entry: ModelEntry):
    for p in [OXFORD_REPO, OXFORD_SITE]:
        if p and os.path.isdir(p) and p not in sys.path:
            sys.path.insert(0, p)
    if not os.path.isfile(OXFORD_SAM_CKPT):
        raise FileNotFoundError(OXFORD_SAM_CKPT)
    if not os.path.isfile(OXFORD_MED_PRETRAIN):
        raise FileNotFoundError(OXFORD_MED_PRETRAIN)
    GlobalHydra.instance().clear()
    from sam2_train.build_sam import build_sam2_video_predictor
    hydra_overrides_extra = [
        f"model.image_size={TARGET_HW}",
        "model.memory_attention.layer.self_attention.feat_sizes=[16,16]",
        "model.memory_attention.layer.cross_attention.feat_sizes=[16,16]",
    ]
    predictor = build_sam2_video_predictor(
        config_file=entry.config_path,
        ckpt_path=OXFORD_SAM_CKPT,
        device=str(DEVICE),
        mode="train",
        hydra_overrides_extra=hydra_overrides_extra,
    )
    if hasattr(predictor, "fill_hole_area"):
        predictor.fill_hole_area = 0
    if hasattr(getattr(predictor, "model", None), "fill_hole_area"):
        predictor.model.fill_hole_area = 0
    target = predictor.model if hasattr(predictor, "model") else predictor
    ckpt = torch.load(OXFORD_MED_PRETRAIN, map_location="cpu")
    sd = unwrap_state_dict(ckpt)
    if any(str(k).startswith("module.") for k in sd.keys()):
        sd = {str(k).replace("module.", "", 1): v for k, v in sd.items()}
    missing, unexpected = target.load_state_dict(sd, strict=False)
    target.to(DEVICE).eval()
    predictor.eval()
    return {"entry": entry, "model": target, "predictor": predictor, "checkpoint": OXFORD_MED_PRETRAIN, "missing": len(missing), "unexpected": len(unexpected)}


def load_uoft_medsam2(entry: ModelEntry):
    for p in [UOFT_MEDSAM2_BASE, UOFT_MEDSAM2_SITE]:
        if p and os.path.isdir(p) and p not in sys.path:
            sys.path.insert(0, p)
    if not os.path.isfile(UOFT_MEDSAM2_CKPT):
        raise FileNotFoundError(UOFT_MEDSAM2_CKPT)
    config_dir = os.path.join(UOFT_MEDSAM2_BASE, "sam2", "configs")
    if not os.path.isdir(config_dir):
        raise FileNotFoundError(config_dir)
    GlobalHydra.instance().clear()
    initialize_config_dir(config_dir=config_dir, version_base=None)
    from sam2.build_sam import build_sam2
    model = build_sam2(entry.config_path)
    ckpt = torch.load(UOFT_MEDSAM2_CKPT, map_location="cpu")
    sd = unwrap_state_dict(ckpt)
    missing, unexpected = model.load_state_dict(sd, strict=False)
    model.to(DEVICE).eval()
    return {"entry": entry, "model": model, "checkpoint": UOFT_MEDSAM2_CKPT, "missing": len(missing), "unexpected": len(unexpected)}


def load_model(entry: ModelEntry):
    if entry.builder in ("rwkv_student", "sam21_base_student"):
        return load_rwkv_model(entry)
    if entry.builder == "oxford_video_predictor":
        return load_oxford_model(entry)
    if entry.builder == "uoft_medsam2_build_sam2":
        return load_uoft_medsam2(entry)
    raise ValueError(f"Unknown builder: {entry.builder}")


## 5. Forward Adapters

All adapters return logits shaped `[T,H,W]`.

In [ ]:
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406], dtype=torch.float32).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225], dtype=torch.float32).view(1, 3, 1, 1)


def normalize_img_0_1(img_bchw):
    return (img_bchw - IMAGENET_MEAN.to(img_bchw.device)) / IMAGENET_STD.to(img_bchw.device)


def forward_direct_sam_prompt(model, imgs_t3hw_01, prompt_plan, normalize_imagenet=False):
    model.eval()
    imgs = imgs_t3hw_01.to(DEVICE, non_blocking=True).float()
    if normalize_imagenet:
        imgs_in = normalize_img_0_1(imgs)
    else:
        imgs_in = imgs
    T, _, H, W = imgs.shape
    with torch.inference_mode():
        backbone_out = model.forward_image(imgs_in)
        _, vision_feats, vision_pos_embeds, feat_sizes = model._prepare_backbone_features(backbone_out)
        feats = [feat.permute(1, 2, 0).reshape(T, -1, *sz) for feat, sz in zip(vision_feats[::-1], feat_sizes[::-1])][::-1]
        image_embed = feats[-1]
        hires_feats = feats[:-1]
        mode = prompt_plan["effective_prompt_mode"]
        if mode == "box":
            boxes = torch.as_tensor(prompt_plan["boxes"], dtype=torch.float32, device=DEVICE).view(T, 2, 2)
            sparse, dense = model.sam_prompt_encoder(points=None, boxes=boxes, masks=None)
        elif mode == "mask":
            masks = torch.as_tensor(prompt_plan["mask_prompts"], dtype=torch.float32, device=DEVICE).view(T, 1, H, W)
            sparse, dense = model.sam_prompt_encoder(points=None, boxes=None, masks=masks)
        else:
            raise ValueError(mode)
        if dense.shape[-2:] != image_embed.shape[-2:]:
            dense = F.interpolate(dense, size=image_embed.shape[-2:], mode="bilinear", align_corners=False)
        image_pe = model.sam_prompt_encoder.get_dense_pe()
        if image_pe.shape[-2:] != image_embed.shape[-2:]:
            image_pe = F.interpolate(image_pe, size=image_embed.shape[-2:], mode="bilinear", align_corners=False)
        image_pe = image_pe.to(image_embed.dtype)
        lowres_logits, _, _, _ = model.sam_mask_decoder(
            image_embeddings=image_embed,
            image_pe=image_pe,
            sparse_prompt_embeddings=sparse,
            dense_prompt_embeddings=dense,
            multimask_output=False,
            repeat_image=False,
            high_res_features=hires_feats,
        )
        logits = F.interpolate(lowres_logits, size=(H, W), mode="bilinear", align_corners=False)
        return logits[:, 0].detach().cpu().numpy().astype(np.float32)


def forward_video_predictor(predictor, imgs_t3hw_01, prompt_plan):
    predictor.eval()
    imgs = imgs_t3hw_01.to(DEVICE, non_blocking=True).float()
    T, _, H, W = imgs.shape
    mode = prompt_plan["effective_prompt_mode"]
    with torch.inference_mode():
        if hasattr(predictor, "init_state_from_tensor"):
            state = predictor.init_state_from_tensor(imgs_tensor=imgs, offload_video_to_cpu=False, offload_state_to_cpu=False, mode="eval")
            add_box = getattr(predictor, "add_new_points_or_box")
            add_mask = getattr(predictor, "add_new_mask")
            propagate = getattr(predictor, "propagate_in_video")
            for t in prompt_plan["prompt_frames"]:
                if mode == "box":
                    box = torch.as_tensor(prompt_plan["boxes"][t], dtype=torch.float32, device=DEVICE)
                    empty_points = torch.empty((1, 0, 2), dtype=torch.float32, device=DEVICE)
                    empty_labels = torch.empty((1, 0), dtype=torch.int32, device=DEVICE)
                    add_box(state, int(t), 1, points=empty_points, labels=empty_labels, box=box, clear_old_points=True, normalize_coords=True)
                elif mode == "mask":
                    mask = torch.as_tensor(prompt_plan["mask_prompts"][t] > 0, dtype=torch.bool, device=DEVICE)
                    add_mask(state, int(t), 1, mask=mask)
                else:
                    raise ValueError(mode)
            logits = torch.zeros((T, H, W), device=DEVICE, dtype=torch.float32)
            for frame_idx, obj_ids, video_res_masks in propagate(state, start_frame_idx=0, max_frame_num_to_track=T, reverse=False):
                logits[int(frame_idx)] = video_res_masks[0, 0].float()
            return logits.detach().cpu().numpy().astype(np.float32)
        if hasattr(predictor, "train_init_state"):
            state = predictor.train_init_state(imgs_tensor=imgs)
            for t in prompt_plan["prompt_frames"]:
                if mode == "box":
                    box = torch.as_tensor(prompt_plan["boxes"][t], dtype=torch.float32, device=DEVICE)
                    predictor.train_add_new_bbox(state, int(t), 0, box)
                elif mode == "mask":
                    mask = torch.as_tensor(prompt_plan["mask_prompts"][t] > 0, dtype=torch.bool, device=DEVICE)
                    predictor.train_add_new_mask(state, int(t), 0, mask)
                else:
                    raise ValueError(mode)
            logits = torch.zeros((T, H, W), device=DEVICE, dtype=torch.float32)
            for out_fid, out_obj_ids, out_mask_logits in predictor.train_propagate_in_video(state, start_frame_idx=0):
                logits[int(out_fid)] = out_mask_logits[0, 0].float()
            try:
                predictor.reset_state(state)
            except Exception:
                pass
            return logits.detach().cpu().numpy().astype(np.float32)
    raise RuntimeError(f"Unsupported predictor API: {type(predictor)}")


def predict_logits(model_bundle, case, prompt_plan):
    entry = model_bundle["entry"]
    imgs = case["image"]
    if entry.builder == "uoft_medsam2_build_sam2":
        return forward_direct_sam_prompt(model_bundle["model"], imgs, prompt_plan, normalize_imagenet=True)
    if entry.builder == "oxford_video_predictor":
        # Prefer the prompt-decoder path from Oxford-Refine_v9 for mask support.
        return forward_direct_sam_prompt(model_bundle["model"], imgs, prompt_plan, normalize_imagenet=False)
    return forward_video_predictor(model_bundle["model"], imgs, prompt_plan)


def parameter_count(model_bundle):
    model = model_bundle.get("model", None)
    if model is None or not hasattr(model, "parameters"):
        return float("nan")
    return int(sum(p.numel() for p in model.parameters()))


def timed_predict(model_bundle, case, prompt_plan):
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    logits = predict_logits(model_bundle, case, prompt_plan)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t1 = time.perf_counter()
    peak_alloc = torch.cuda.max_memory_allocated() if torch.cuda.is_available() else 0
    peak_reserved = torch.cuda.max_memory_reserved() if torch.cuda.is_available() else 0
    elapsed = float(t1 - t0)
    frames = int(logits.shape[0])
    return logits, {
        "wall_time_sec": elapsed,
        "cuda_time_sec": elapsed,
        "peak_cuda_mem_allocated_mb": peak_alloc / (1024**2),
        "peak_cuda_mem_reserved_mb": peak_reserved / (1024**2),
        "frames_per_sec": frames / elapsed if elapsed > 0 else float("nan"),
        "cases_per_sec": 1.0 / elapsed if elapsed > 0 else float("nan"),
        "n_frames": frames,
        "param_count": parameter_count(model_bundle),
    }


## 6. Preflight

This cell checks model availability and runs a mini prediction for available models.

In [ ]:
def get_smoke_case():
    ds = benchmark_loader.dataset
    by_dim = first_cases_by_dim(benchmark_loader, 1)
    idx = None
    for dim in [3, 2]:
        if by_dim.get(dim):
            idx = by_dim[dim][0]
            break
    if idx is None:
        idx = 0
    return sample_to_case(ds[idx], dataset_index=idx)


def preflight_registry(registry):
    rows = []
    loaded = {}
    smoke = get_smoke_case()
    for entry in registry:
        row = asdict(entry).copy()
        row.update({
            "config_found": bool((entry.config_path and os.path.exists(entry.config_path)) or entry.family in ("oxford", "uoft")),
            "checkpoint_found": bool(pick_checkpoint(entry)),
            "cuda_available": bool(torch.cuda.is_available()),
            "build_ok": False,
            "mini_inference_ok": False,
            "error": "",
        })
        if not entry.enabled:
            row["status"] = "disabled"
            rows.append(row)
            continue
        if not row["checkpoint_found"]:
            row["status"] = "pending"
            rows.append(row)
            continue
        try:
            bundle = load_model(entry)
            row["build_ok"] = True
            plan = prompt_plan_for_case(smoke, "box")
            logits, eff = timed_predict(bundle, smoke, plan)
            row["mini_inference_ok"] = bool(logits.shape == smoke["gt_thw"].shape)
            row["status"] = "available" if row["mini_inference_ok"] else "shape_failed"
            row.update({k: eff[k] for k in ["wall_time_sec", "peak_cuda_mem_allocated_mb", "param_count"]})
            loaded[entry.name] = bundle
        except Exception as e:
            row["status"] = "failed"
            row["error"] = repr(e)
            traceback.print_exc(limit=2)
        rows.append(row)
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()
    return rows, loaded

preflight_rows, LOADED_MODELS = preflight_registry(MODEL_REGISTRY)
write_table(preflight_rows, OUT_DIR / "preflight.csv")
if pd is not None:
    display(pd.DataFrame(preflight_rows))
else:
    preflight_rows


## 7. Smoke Test

In [ ]:
def iter_smoke_cases():
    ds = benchmark_loader.dataset
    by_dim = first_cases_by_dim(benchmark_loader, SMOKE_CASES_PER_DIM)
    for dim, idxs in by_dim.items():
        for idx in idxs:
            yield sample_to_case(ds[idx], dataset_index=idx)

smoke_records = []
for model_name, bundle in LOADED_MODELS.items():
    for case in iter_smoke_cases():
        for prompt_mode in PROMPT_MODES:
            plan = prompt_plan_for_case(case, prompt_mode)
            logits, eff = timed_predict(bundle, case, plan)
            metrics = segmentation_metrics_from_logits(logits, case["gt_thw"])
            row = {**case["meta"], "model": model_name, "prompt_mode": prompt_mode, **metrics, **eff}
            smoke_records.append(row)
            print("SMOKE", model_name, prompt_mode, case["meta"], "dice", row["dice"], "iou", row["iou"], "time", row["wall_time_sec"])
write_table(smoke_records, OUT_DIR / "smoke_results.csv")


## 8. Full Benchmark

In [ ]:
def iter_benchmark_cases(loader, max_cases=MAX_TEST_CASES):
    ds = loader.dataset
    n = len(ds) if max_cases is None else min(len(ds), int(max_cases))
    for idx in range(n):
        yield sample_to_case(ds[idx], dataset_index=idx)

raw_records = []
jsonl_path = OUT_DIR / "per_case_results.jsonl"
with open(jsonl_path, "w") as jf:
    for model_name, bundle in LOADED_MODELS.items():
        for case in tqdm(iter_benchmark_cases(benchmark_loader), desc=f"Benchmark {model_name}"):
            for prompt_mode in PROMPT_MODES:
                plan = prompt_plan_for_case(case, prompt_mode)
                try:
                    logits, eff = timed_predict(bundle, case, plan)
                    metrics = segmentation_metrics_from_logits(logits, case["gt_thw"])
                    row = {
                        **case["meta"],
                        "model": model_name,
                        "prompt_mode": prompt_mode,
                        "effective_prompt_mode": plan["effective_prompt_mode"],
                        "prompt_frames": ";".join(map(str, plan["prompt_frames"])),
                        **metrics,
                        **eff,
                        "status": "ok",
                    }
                except Exception as e:
                    row = {**case["meta"], "model": model_name, "prompt_mode": prompt_mode, "status": "failed", "error": repr(e)}
                    traceback.print_exc(limit=2)
                raw_records.append(row)
                jf.write(json.dumps(row) + "\n")
                jf.flush()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

ok_records = [r for r in raw_records if r.get("status") == "ok"]
write_table(raw_records, OUT_DIR / "per_case_results.csv")
summary_model = aggregate_records(ok_records, ["model"])
summary_model_prompt = aggregate_records(ok_records, ["model", "prompt_mode"])
summary_task = aggregate_records(ok_records, ["model", "prompt_mode", "task_id"])
summary_modality = aggregate_records(ok_records, ["model", "prompt_mode", "modality"])
summary_modality_task = aggregate_records(ok_records, ["model", "prompt_mode", "modality", "task_id"])
efficiency = aggregate_records(ok_records, ["model", "prompt_mode"])
threshold_sweep_task = threshold_sweep_records(ok_records, ["model", "prompt_mode", "task_id"])
threshold_sweep_modality = threshold_sweep_records(ok_records, ["model", "prompt_mode", "modality"])
threshold_sweep_modality_task = threshold_sweep_records(ok_records, ["model", "prompt_mode", "modality", "task_id"])

write_table(summary_model, OUT_DIR / "summary_by_model.csv")
write_table(summary_model_prompt, OUT_DIR / "summary_by_model_prompt.csv")
write_table(summary_task, OUT_DIR / "summary_by_model_prompt_task.csv")
write_table(summary_modality, OUT_DIR / "summary_by_model_prompt_modality.csv")
write_table(summary_modality_task, OUT_DIR / "summary_by_model_prompt_modality_task.csv")
write_table(efficiency, OUT_DIR / "efficiency_by_model_prompt.csv")
write_table(threshold_sweep_task, OUT_DIR / "threshold_sweep_by_model_prompt_task.csv")
write_table(threshold_sweep_modality, OUT_DIR / "threshold_sweep_by_model_prompt_modality.csv")
write_table(threshold_sweep_modality_task, OUT_DIR / "threshold_sweep_by_model_prompt_modality_task.csv")

if pd is not None:
    display(pd.DataFrame(summary_model_prompt).sort_values(["prompt_mode", "dice"], ascending=[True, False]))


## 9. Optional Visual Comparisons

This saves a few static overlays. GIF export can be added later if desired; the raw per-case rows already contain the selected case ids and prompt frames.

In [ ]:
try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None


def save_overlay_png(image_hw, gt_hw, pred_hw, path, title=""):
    if plt is None:
        return
    img = np.asarray(image_hw)
    if img.ndim == 3:
        img = np.moveaxis(img, 0, -1)
        img = np.clip(img, 0, 1)
    fig, ax = plt.subplots(1, 1, figsize=(5, 5))
    ax.imshow(img, cmap="gray")
    ax.contour(gt_hw.astype(bool), colors="lime", linewidths=1)
    ax.contour(pred_hw.astype(bool), colors="red", linewidths=1)
    ax.set_title(title)
    ax.axis("off")
    fig.tight_layout()
    fig.savefig(path, dpi=140)
    plt.close(fig)

VIS_DIR = OUT_DIR / "visual_examples"
VIS_DIR.mkdir(exist_ok=True, parents=True)

# Regenerate predictions for the first few available cases to keep visualization independent of raw logits storage.
if LOADED_MODELS and plt is not None:
    for case_i, case in enumerate(iter_benchmark_cases(benchmark_loader, max_cases=3)):
        t = int(np.argmax(case["gt_thw"].reshape(case["gt_thw"].shape[0], -1).sum(axis=1)))
        for model_name, bundle in LOADED_MODELS.items():
            for prompt_mode in PROMPT_MODES:
                plan = prompt_plan_for_case(case, prompt_mode)
                try:
                    logits, _ = timed_predict(bundle, case, plan)
                    pred = sigmoid_np(logits[t]) >= PRIMARY_THRESHOLD
                    img = case["image"][t].detach().cpu().numpy()
                    save_overlay_png(
                        img,
                        case["gt_thw"][t],
                        pred,
                        VIS_DIR / f"case{case_i:03d}_{model_name}_{prompt_mode}_slice{t:03d}.png",
                        title=f"{model_name} | {prompt_mode} | slice {t}",
                    )
                except Exception as e:
                    print("visual failed", case_i, model_name, prompt_mode, repr(e))
print("Visual dir:", VIS_DIR)
